# Kaggle Run CV

Run this notebook after the full notebook if you want ablations and CV.

This notebook does:
1. install dependencies
2. export baseline artifacts from the trained Qwen classifier
3. build KB and graph
4. run cross-validation / ablations
5. show CV summary files


## Attach These In Kaggle Before Running

Attach these datasets/files to the notebook:

1. **CodeDs.zip or extracted CodeDs dataset**
- should contain:
  - `run_experiment.py`
  - `src/`
  - `configs/`
  - `requirements.txt`
  - preferably `kaggle_notebooks/` too
- this notebook now auto-extracts `CodeDs.zip` if you uploaded the repo as a zip

2. **MAP data**
- either attach a separate dataset containing:
  - `DS/train.csv`
  - `DS/test.csv`
  - optional `DS/misconception_mapping.csv`
- or keep those files inside the uploaded code dataset if you prefer

3. **loaded_models dataset**
- should contain:
  - `loaded_models/Qwen3-8B/` for the classifier base model
  - `loaded_models/Qwen2.5-7B-Instruct/` for KB/counterfactual generation
- optional alternative:
  - `loaded_models/Hunyuan-7B-Instruct/`

4. **qwen3-8b-qlora dataset**
- should contain:
  - `adapter/`
- recommended if available:
  - `label_encoder.joblib`
  - `run_config.json`

The notebook now tries to auto-detect these paths and writes a runtime config in `/kaggle/working/runtime_kaggle_h100_qwen_full.yaml`.


In [ ]:
import os
import sys
import zipfile
import subprocess
from pathlib import Path

import yaml


def run(cmd):
    print('$', ' '.join(str(x) for x in cmd))
    subprocess.run([str(x) for x in cmd], check=True)


def looks_like_project_root(path: Path) -> bool:
    return path.is_dir() and (path / 'run_experiment.py').exists() and (path / 'src').exists() and (path / 'configs').exists()


def maybe_extract_codeds_zip() -> Path | None:
    search_roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    extract_root = Path('/kaggle/working/codeds_extracted')
    for root in search_roots:
        if not root.exists():
            continue
        for zip_path in root.rglob('*'):
            if not zip_path.is_file():
                continue
            if zip_path.name.lower() != 'codeds.zip':
                continue
            extract_root.mkdir(parents=True, exist_ok=True)
            if not any(extract_root.iterdir()):
                print('Extracting', zip_path, 'to', extract_root)
                with zipfile.ZipFile(zip_path, 'r') as zf:
                    zf.extractall(extract_root)
            for candidate in [extract_root] + [p for p in extract_root.rglob('*') if p.is_dir()]:
                if looks_like_project_root(candidate):
                    return candidate
    return None


def find_project_root() -> Path:
    direct_candidates = [
        Path.cwd(),
        Path('/kaggle/working'),
        Path('/kaggle/input'),
    ]
    for base in direct_candidates:
        if looks_like_project_root(base):
            return base

    extracted = maybe_extract_codeds_zip()
    if extracted is not None:
        return extracted

    for parent in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if not parent.exists():
            continue
        for candidate in parent.rglob('*'):
            if candidate.is_dir() and looks_like_project_root(candidate):
                return candidate

    raise FileNotFoundError('Could not find the project root. Attach CodeDs.zip or an extracted repo dataset containing run_experiment.py, src/, and configs/.')


def find_first_file(candidates):
    for candidate in candidates:
        path = Path(candidate)
        if path.exists():
            return path
    return None


def find_first_matching_dir(dir_names, required_file='config.json'):
    search_roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    for root in search_roots:
        if not root.exists():
            continue
        for dir_name in dir_names:
            for candidate in root.rglob(dir_name):
                if candidate.is_dir() and (candidate / required_file).exists():
                    return candidate
    return None


def find_first_matching_file(path_suffixes):
    search_roots = [Path('/kaggle/input'), Path('/kaggle/working')]
    suffixes = [suffix.replace('\\', '/').lower() for suffix in path_suffixes]
    for root in search_roots:
        if not root.exists():
            continue
        for candidate in root.rglob('*'):
            if not candidate.exists():
                continue
            normalized = str(candidate).replace('\\', '/').lower()
            if any(normalized.endswith(suffix) for suffix in suffixes):
                return candidate
    return None


def build_runtime_config(project_root: Path) -> Path:
    base_config_path = project_root / 'configs' / 'kaggle_h100_qwen_full.yaml'
    config = yaml.safe_load(base_config_path.read_text(encoding='utf-8'))

    train_path = find_first_matching_file([
        'DS/train.csv',
        'train.csv',
    ])
    test_path = find_first_matching_file([
        'DS/test.csv',
        'test.csv',
    ])
    misconception_path = find_first_matching_file([
        'DS/misconception_mapping.csv',
        'misconception_mapping.csv',
    ])

    base_model_path = find_first_matching_dir(['Qwen3-8B'])
    instruct_model_path = find_first_matching_dir(['Qwen2.5-7B-Instruct', 'Hunyuan-7B-Instruct', 'Qwen3-8B'])
    adapter_path = find_first_matching_file([
        'qwen3-8b-qlora/adapter',
        'adapter',
    ])
    label_encoder_path = find_first_matching_file([
        'qwen3-8b-qlora/label_encoder.joblib',
        'label_encoder.joblib',
    ])
    run_config_path = find_first_matching_file([
        'qwen3-8b-qlora/run_config.json',
        'run_config.json',
    ])

    if train_path is not None:
        config['paths']['train_file'] = [str(train_path)]
    if test_path is not None:
        config['paths']['test_file'] = [str(test_path)]
    if misconception_path is not None:
        config['paths']['misconception_file'] = [str(misconception_path)]

    if base_model_path is not None:
        config['baseline_qwen']['base_model_candidates'] = [str(base_model_path)] + list(config['baseline_qwen'].get('base_model_candidates', []))
    if adapter_path is not None:
        config['baseline_qwen']['adapter_candidates'] = [str(adapter_path)] + list(config['baseline_qwen'].get('adapter_candidates', []))
    if label_encoder_path is not None:
        config['baseline_qwen']['label_encoder_candidates'] = [str(label_encoder_path)] + list(config['baseline_qwen'].get('label_encoder_candidates', []))
    if run_config_path is not None:
        config['baseline_qwen']['run_config_candidates'] = [str(run_config_path)] + list(config['baseline_qwen'].get('run_config_candidates', []))
    if instruct_model_path is not None:
        config['llm']['model_candidates'] = [str(instruct_model_path)] + list(config['llm'].get('model_candidates', []))

    runtime_path = Path('/kaggle/working/runtime_kaggle_h100_qwen_full.yaml')
    runtime_path.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')

    print('PROJECT_ROOT =', project_root)
    print('RUNTIME_CONFIG_PATH =', runtime_path)
    print('Detected train_path =', train_path)
    print('Detected test_path =', test_path)
    print('Detected misconception_path =', misconception_path)
    print('Detected base_model_path =', base_model_path)
    print('Detected instruct_model_path =', instruct_model_path)
    print('Detected adapter_path =', adapter_path)
    print('Detected label_encoder_path =', label_encoder_path)
    print('Detected run_config_path =', run_config_path)
    return runtime_path


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
RUNTIME_CONFIG_PATH = build_runtime_config(PROJECT_ROOT)
print('Python =', sys.executable)


In [ ]:
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])


In [ ]:
run([sys.executable, '-m', 'src.baseline.qwen_adapter_infer', '--config', str(RUNTIME_CONFIG_PATH), '--split', 'both'])


In [ ]:
run([sys.executable, '-m', 'src.ontology.build_kb', '--config', str(RUNTIME_CONFIG_PATH), '--output', '/kaggle/working/map_mvp/data/misconception_kb.json'])


In [ ]:
run([sys.executable, '-m', 'src.ontology.build_graph', '--config', str(RUNTIME_CONFIG_PATH)])


In [ ]:
run([sys.executable, 'run_experiment.py', '--config', str(RUNTIME_CONFIG_PATH), '--run-cv'])


In [ ]:
from pathlib import Path
import pandas as pd

output_dir = Path('/kaggle/working/map_mvp/output')
for name in ['cv_metrics.csv', 'ablation_summary.csv']:
    path = output_dir / name
    print(path)
    if path.exists():
        display(pd.read_csv(path).head())
